# 2주차 2교시 · CSV 25개를 하나의 활동 기록표로 합치기

**학생용 지적실험노트**

오늘의 질문: **파일을 모두 합친 뒤에도 각 행이 어느 CSV에서 왔는지 알려면 무엇을 남겨야 할까요?**

오늘의 완성 결과: 원본 파일명이 적힌 source_file 열과 30,000행×10열 activity DataFrame

각 Q에서 ① 실행 전 예상 ② ___ 채우기 ③ 실행 ④ 값 하나 바꾸기 ⑤ 결과 이유 한 문장을 지킵니다. `NameError: ___`는 고장이 아니라 아직 빈칸이 남았다는 뜻입니다.


In [ ]:
from pathlib import Path
import pandas as pd

def find_week2_data_dir():
    """노트북을 어느 교시 폴더에서 열어도 2주차 데이터 폴더를 찾습니다."""
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:
        candidates = [
            base / "course_materials" / "matchmaking_data" / "02_week2",
            base / "matchmaking_data" / "02_week2",
            base / "02_week2",
        ]
        for candidate in candidates:
            if (candidate / "activity_batches").exists():
                return candidate
    raise FileNotFoundError("02_week2 데이터 폴더를 찾지 못했습니다. python_lecture 프로젝트 안에서 노트북을 열어 주세요.")

DATA_DIR = find_week2_data_dir()
print("데이터 폴더:", DATA_DIR)


## Q1. 첫 CSV를 열어 1,200행×9열인지 확인합니다

**먼저 예상하기**

- 출력된 1,200은 회원 수일까요, 하루 활동 기록의 수일까요?
- 내 예상: ____________________

**왜 이 일을 하나요?** 25개를 한꺼번에 합치기 전에 첫 파일이 예상한 크기와 열 이름을 가지고 있는지 확인합니다.

**현재 자료구조** batch_files[0]은 첫 CSV의 위치를 담은 Path입니다. pd.read_csv가 그 파일을 읽으면 first_batch라는 1,200행×9열 DataFrame이 생깁니다.

**코드 읽기 도움** pd.read_csv는 Path가 가리키는 CSV를 읽어 행과 열이 있는 DataFrame으로 만듭니다.

**정상 결과** (1200, 9)와 9개 열 이름

**오류가 나면** FileNotFoundError가 나오거나 열 수가 9개가 아니라면 batch_files[0].name을 출력해 activity_batch_01.csv인지 보고, 1교시에서 batch_files를 만든 셀부터 다시 실행합니다.


In [ ]:
batch_files = sorted((DATA_DIR / "activity_batches").glob("activity_batch_*.csv"))
first_batch = pd.___(batch_files[0])
print(first_batch.shape)
print(first_batch.columns.tolist())

## Q2. 각 행에 원본 CSV의 파일명을 기록합니다

**먼저 예상하기**

- source_file 열을 하나 추가하면 열 수는 9개에서 몇 개가 될까요?
- 내 예상: ____________________

**왜 이 일을 하나요?** 25개 파일을 합친 뒤 이상한 값이 발견되어도 어느 원본 파일에서 왔는지 찾을 수 있게 합니다.

**현재 자료구조** frames는 빈 list에서 시작합니다. for가 앞의 CSV 세 개를 하나씩 읽고, source_file 열을 추가한 1,200행×10열 DataFrame을 frames에 한 개씩 넣습니다.

**코드 읽기 도움** file_path.name은 현재 CSV의 파일명만 꺼냅니다. append는 source_file 열까지 추가된 DataFrame 한 개를 frames에 항목 하나로 넣습니다.

**정상 결과** 3 (1200, 10)

**오류가 나면** frames가 비어 있거나 source_file 열에 같은 파일명만 들어 있다면 source_file을 만드는 줄과 frames.append 줄이 for 아래에 같은 들여쓰기로 들어갔는지 확인합니다.


In [ ]:
frames = []
for file_path in batch_files[:3]:
    one_batch = pd.read_csv(file_path)
    one_batch["source_file"] = ___
    frames.___(one_batch)
print(len(frames), frames[0].shape)

## Q3. 먼저 CSV 세 개를 합쳐 3,600행을 만듭니다

**먼저 예상하기**

- 1,200행인 표 세 개를 위아래로 붙이면 sample_activity는 몇 행이 될까요?
- 내 예상: ____________________

**왜 이 일을 하나요?** 전체 25개에 적용하기 전에 작은 세 파일로 표가 위아래로 제대로 합쳐지는지 확인합니다.

**현재 자료구조** frames에는 1,200행×10열 DataFrame 세 개가 들어 있습니다. pd.concat이 세 표를 이어 sample_activity라는 3,600행×10열 DataFrame을 새로 만듭니다.

**코드 읽기 도움** pd.concat은 list 안의 DataFrame을 위아래로 잇습니다. ignore_index=True는 member_id를 지우지 않고 화면 왼쪽의 행 번호를 0부터 3599까지 새로 붙입니다.

**정상 결과** (3600, 10), [0, 3599]

**오류가 나면** sample_activity가 3,600행이 아니거나 마지막 index가 3599가 아니라면 len(frames)가 3인지 먼저 보고, pd.concat의 입력이 frames인지와 ignore_index=True인지 확인합니다.


In [ ]:
sample_activity = pd.___(frames, ignore_index=___)
print(sample_activity.shape)
print(sample_activity.index[[0, -1]].tolist())

## Q4. 연습 코드를 CSV 25개 전체에 적용합니다

**먼저 예상하기**

- 각 파일이 1,200행이라면 25개를 모두 합친 activity는 몇 행이 될까요?
- 내 예상: ____________________

**왜 이 일을 하나요?** 앞 세 파일에서 확인한 읽기, 출처 기록, 결합 과정을 전체 파일에 반복합니다.

**현재 자료구조** batch_files는 Path 25개가 든 list입니다. 반복이 끝나면 frames에는 DataFrame 25개가 있고, activity에는 30,000행×10열이 들어 있습니다.

**코드 읽기 도움** nunique()는 source_file 열에 서로 다른 파일명이 몇 종류인지 셉니다. 25가 나오면 파일 25개의 이름이 모두 들어갔다는 뜻입니다.

**정상 결과** (30000, 10) 25

**오류가 나면** 행 수가 3,600이거나 source_file의 파일명이 3종뿐이라면 for 뒤의 batch_files[:3]에서 [:3]을 빼고 batch_files를 사용했는지 확인합니다.


In [ ]:
frames = []
for file_path in ___:
    one_batch = pd.read_csv(file_path)
    one_batch["source_file"] = file_path.name
    frames.append(one_batch)
activity = pd.concat(frames, ignore_index=True)
print(activity.shape, activity["source_file"].___())

## Q5. 원본의 전체 행 수와 합친 표의 행 수를 비교합니다

**먼저 예상하기**

- source_rows와 len(activity)가 모두 30,000이면 무엇을 확인한 것일까요?
- 내 예상: ____________________

**왜 이 일을 하나요?** 파일 하나가 빠지거나 두 번 들어간 실수를 숫자로 확인합니다.

**현재 자료구조** source_rows는 원본 CSV 25개의 행 수를 더한 정수이고, len(activity)는 합친 DataFrame의 행 수를 나타내는 정수입니다.

**코드 읽기 도움** assert는 뒤의 조건이 True면 조용히 지나가고 False면 멈춥니다. 두 번째 assert는 원본 파일명이 25종인지 검사합니다.

**정상 결과** 30000 30000

**오류가 나면** AssertionError가 나오거나 두 숫자가 다르면 len(batch_files), len(frames), activity["source_file"].nunique()를 차례로 출력해 모두 25인지 확인합니다.


In [ ]:
source_rows = sum(len(pd.read_csv(p)) for p in batch_files)
assert len(activity) == ___
assert activity["source_file"].nunique() == ___
print(source_rows, len(activity))

## 마무리 확인

- [ ] 오늘 결과가 **원본 파일명이 적힌 source_file 열과 30,000행×10열 activity DataFrame**과 같은가?
- [ ] 실행 순서를 자기 말로 설명할 수 있는가?
- [ ] 오류가 났을 때 마지막 줄과 직전 중간 결과를 확인했는가?
